# Sentiment Analysis of Customer Reviews
## Exploratory Data Analysis & Model Training Notebook

**Author:** Your Name  
**Dataset:** Restaurant Reviews  
**Goal:** Classify customer reviews as Positive (1) or Negative (0)

---

## 1. Setup & Imports

In [ ]:
import os, re, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

for pkg in ['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

print('Setup complete!')

## 2. Load Dataset

In [ ]:
# Run train.py first to generate data/Restaurant_Reviews.tsv
df = pd.read_csv('../data/Restaurant_Reviews.tsv', sep='\t', quoting=3)
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
print('Sentiment distribution:')
print(df['Liked'].value_counts())
print(f'\nMissing values:\n{df.isnull().sum()}')

## 3. Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def preprocess_text(text):
    text   = text.lower()
    text   = re.sub(r'http\S+|www\S+', '', text)
    text   = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df['cleaned_review'] = df['Review'].apply(preprocess_text)

# Show before/after
for i in range(3):
    print(f'ORIGINAL : {df["Review"].iloc[i]}')
    print(f'CLEANED  : {df["cleaned_review"].iloc[i]}')
    print()

## 4. Exploratory Data Analysis

In [ ]:
# Sentiment distribution
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['Liked'].value_counts()
ax.bar(['Negative', 'Positive'], counts.values, color=['#e74c3c', '#2ecc71'])
ax.set_title('Sentiment Distribution', fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Top-20 words
all_words = ' '.join(df['cleaned_review']).split()
top20 = Counter(all_words).most_common(20)
words, freqs = zip(*top20)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(words[::-1], freqs[::-1], color='#3498db')
ax.set_title('Top 20 Words (After Preprocessing)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top positive vs negative words side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, colour, title in [
    (axes[0], 1, '#2ecc71', 'Top 15 Positive Words'),
    (axes[1], 0, '#e74c3c', 'Top 15 Negative Words'),
]:
    words_raw = ' '.join(df[df['Liked'] == label]['cleaned_review']).split()
    top = Counter(words_raw).most_common(15)
    w, f = zip(*top)
    ax.barh(w[::-1], f[::-1], color=colour)
    ax.set_title(title, fontweight='bold')

plt.suptitle('Word Frequency by Sentiment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. TF-IDF Vectorisation & Model Training

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, sublinear_tf=True)
X = tfidf.fit_transform(df['cleaned_review'])
y = df['Liked'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Naive Bayes':         MultinomialNB(alpha=0.5),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1 Score':  round(f1_score(y_test, y_pred, zero_division=0), 4),
    })
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=['Negative','Positive']))

pd.DataFrame(results)

## 6. Model Comparison Chart

In [ ]:
results_df  = pd.DataFrame(results)
metrics     = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x           = np.arange(len(metrics))
width       = 0.22
colours     = ['#3498db', '#e67e22', '#9b59b6']

fig, ax = plt.subplots(figsize=(11, 5))
for i, (_, row) in enumerate(results_df.iterrows()):
    ax.bar(x + i*width, [row[m] for m in metrics], width,
           label=row['Model'], color=colours[i], edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()